In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from pathlib import Path

from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

pd.set_option("display.max_columns", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:

    required_file = (
        PROJECT_ROOT /
        "Module_3_Feature_Engineering_Vectorization" /
        "features" /
        "combined_tfidf_matrix.npz"
    )

    if required_file.exists():
        break

    PROJECT_ROOT = PROJECT_ROOT.parent

if not required_file.exists():
    raise FileNotFoundError(
        "Combined TF-IDF matrix not found. "
        "Complete Notebook 02 first."
    )


MODULE_DIR = (
    PROJECT_ROOT /
    "Module_3_Feature_Engineering_Vectorization"
)

FEATURE_DIR = MODULE_DIR / "features"
MODEL_DIR = MODULE_DIR / "models"
SIMILARITY_DIR = MODULE_DIR / "similarity"

OUTPUT_DIR = MODULE_DIR / "outputs"

VIZ_DIR = OUTPUT_DIR / "visualizations"
REPORT_DIR = OUTPUT_DIR / "reports"
STATISTICS_DIR = OUTPUT_DIR / "statistics"

for folder in [
    SIMILARITY_DIR,
    VIZ_DIR,
    REPORT_DIR,
    STATISTICS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


print("Project root:", PROJECT_ROOT)
print("Feature directory:", FEATURE_DIR)
print("Similarity directory:", SIMILARITY_DIR)

FileNotFoundError: Combined TF-IDF matrix not found. Complete Notebook 02 first.

In [ ]:
DATA_PATH = (
    FEATURE_DIR /
    "ted_talks_features.csv"
)

MATRIX_PATH = (
    FEATURE_DIR /
    "combined_tfidf_matrix.npz"
)

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

tfidf_matrix = load_npz(
    MATRIX_PATH
).tocsr()

print("Dataset shape:", df.shape)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

print("Number of talks:", len(df))

In [ ]:
field_names = [
    "title",
    "description",
    "tags",
    "transcript"
]

field_matrices = {}

for field in field_names:

    matrix_path = (
        FEATURE_DIR /
        f"{field}_tfidf_matrix.npz"
    )

    if not matrix_path.exists():

        raise FileNotFoundError(
            f"Missing matrix: {matrix_path}"
        )

    field_matrices[field] = load_npz(
        matrix_path
    ).tocsr()

    print(
        field,
        field_matrices[field].shape
    )

In [ ]:
# Normalise rows to unit length.
normalised_matrix = normalize(
    tfidf_matrix,
    norm="l2",
    axis=1
).tocsr()

print("Normalised matrix:", normalised_matrix.shape)

In [ ]:
def get_similar_talks(
    talk_index,
    matrix,
    dataframe,
    top_n=10
):

    # Similarity between the selected talk and all other talks
    scores = cosine_similarity(
        matrix[talk_index],
        matrix
    ).flatten()

    # Exclude the selected talk itself
    scores[talk_index] = -1

    # Get indices of highest scores
    top_indices = np.argsort(
        scores
    )[::-1][:top_n]

    results = dataframe.iloc[
        top_indices
    ][
        ["title_processed"]
    ].copy()

    results.insert(
        0,
        "talk_index",
        top_indices
    )

    results["similarity_score"] = (
        scores[top_indices]
    )

    return results.reset_index(drop=True)

In [ ]:
sample_index = 0

print("Selected TED Talk:")

print(
    df.iloc[sample_index]["title_processed"]
)

recommendations = get_similar_talks(
    talk_index=sample_index,
    matrix=normalised_matrix,
    dataframe=df,
    top_n=10
)

display(recommendations)

In [ ]:
def generate_all_recommendations(
    matrix,
    dataframe,
    top_n=10
):

    results = []

    for index in range(matrix.shape[0]):

        scores = cosine_similarity(
            matrix[index],
            matrix
        ).flatten()

        # Exclude self-match
        scores[index] = -1

        top_indices = np.argsort(
            scores
        )[::-1][:top_n]

        for rank, recommended_index in enumerate(
            top_indices,
            start=1
        ):

            results.append({

                "talk_index": index,

                "talk_title": dataframe.iloc[
                    index
                ]["title_processed"],

                "recommended_index": int(
                    recommended_index
                ),

                "recommended_title": dataframe.iloc[
                    recommended_index
                ]["title_processed"],

                "rank": rank,

                "similarity_score": float(
                    scores[recommended_index]
                )

            })

    return pd.DataFrame(results)

In [ ]:
recommendation_df = generate_all_recommendations(
    matrix=normalised_matrix,
    dataframe=df,
    top_n=10
)

print("Recommendation generation completed!")

print("Recommendation records:", len(recommendation_df))

display(recommendation_df.head(20))

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    recommendation_df["similarity_score"],
    bins=30
)

plt.title("Distribution of TED Talk Similarity Scores")

plt.xlabel("Cosine Similarity Score")

plt.ylabel("Number of Recommendations")

plt.tight_layout()

plt.show()

In [ ]:
selected_talk = df.iloc[
    sample_index
]["title_processed"]

sample_recommendations = recommendation_df[
    recommendation_df["talk_index"] == sample_index
].sort_values("rank")

plt.figure(figsize=(11, 6))

plt.barh(
    sample_recommendations["recommended_title"][::-1],
    sample_recommendations["similarity_score"][::-1]
)

plt.title(
    f"Top Similar TED Talks\n{selected_talk}"
)

plt.xlabel("Cosine Similarity")

plt.ylabel("Recommended TED Talks")

plt.tight_layout()

plt.show()

In [ ]:
def get_field_similarity(
    talk_index,
    matrices,
    dataframe,
    top_n=5
):

    results = []

    for field, matrix in matrices.items():

        scores = cosine_similarity(
            matrix[talk_index],
            matrix
        ).flatten()

        scores[talk_index] = -1

        top_indices = np.argsort(
            scores
        )[::-1][:top_n]

        for rank, index in enumerate(
            top_indices,
            start=1
        ):

            results.append({

                "field": field,

                "rank": rank,

                "talk_index": talk_index,

                "recommended_index": int(index),

                "recommended_title": dataframe.iloc[
                    index
                ]["title_processed"],

                "similarity_score": float(
                    scores[index]
                )

            })

    return pd.DataFrame(results)

In [ ]:
field_comparison = get_field_similarity(
    talk_index=sample_index,
    matrices=field_matrices,
    dataframe=df,
    top_n=5
)

display(field_comparison)

In [ ]:
inspection = recommendation_df[
    recommendation_df["talk_index"] == sample_index
][
    [
        "talk_title",
        "recommended_title",
        "similarity_score"
    ]
].copy()

display(inspection)

In [ ]:
RECOMMENDATION_PATH = (
    SIMILARITY_DIR /
    "top_similar_talks.csv"
)

recommendation_df.to_csv(
    RECOMMENDATION_PATH,
    index=False,
    encoding="utf-8"
)

FIELD_COMPARISON_PATH = (
    SIMILARITY_DIR /
    "field_similarity_comparison.csv"
)

field_comparison.to_csv(
    FIELD_COMPARISON_PATH,
    index=False,
    encoding="utf-8"
)

print("Recommendation results saved!")
print(RECOMMENDATION_PATH)
print(FIELD_COMPARISON_PATH)

In [ ]:
similarity_statistics = {

    "total_talks": int(len(df)),

    "recommendations_per_talk": 10,

    "total_recommendation_records": int(
        len(recommendation_df)
    ),

    "mean_similarity": float(
        recommendation_df["similarity_score"].mean()
    ),

    "median_similarity": float(
        recommendation_df["similarity_score"].median()
    ),

    "minimum_similarity": float(
        recommendation_df["similarity_score"].min()
    ),

    "maximum_similarity": float(
        recommendation_df["similarity_score"].max()
    )

}

with open(
    REPORT_DIR / "similarity_analysis_report.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        similarity_statistics,
        file,
        indent=4
    )

print("Similarity report saved!")

In [ ]:
files_to_check = [

    SIMILARITY_DIR / "top_similar_talks.csv",

    SIMILARITY_DIR / "field_similarity_comparison.csv",

    REPORT_DIR / "similarity_analysis_report.json"

]

print("FINAL VERIFICATION")

for file in files_to_check:

    print(
        file.name,
        "->",
        "SAVED" if file.exists() else "MISSING"
    )

    if file.exists():

        print(
            "Size:",
            f"{file.stat().st_size:,} bytes"
        )